In [10]:
# Imports
import os
import dotenv
from typing import Any, Dict, List, Optional, TypedDict
import datetime
import json
from abc import ABC, abstractmethod

# Web Scraping
import requests
from bs4 import BeautifulSoup

# 3rd Party APIs
import finnhub

# Image processing
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image

# LLM APIs
from openai import OpenAI

# User interface
import gradio as gr

In [12]:
# Config

# Load environment variables
dotenv.load_dotenv()

# Finnhub Client
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY")
finnhub_client = finnhub.Client(FINNHUB_API_KEY)

# LLM Client
openai = OpenAI()
SONAR_URL = "https://api.perplexity.ai/chat/completions"

# LLM models
OPEN_AI_MODEL = "gpt-4o-mini"
OPEN_AI_IMAGE_MODEL = "dall-e-3"
OPEN_AI_AUDIO_MODEL = "tts-1"
GOOGLE_MODEL = "gemini-1.5-flash-latest"
SONAR_MODEL = "sonar-pro"

## LLM Parameters
IMAGE_SIZE = "1024x1024"
N = 1
RESPONSE_FORMAT = "b64_json"
VOICE = "alloy"
AUDIO_FORMAT = "mp3"

# LLM Instructions
chat_system_message = """
You are a finance assistant that is able to summarize earnings call and report of the public companies and also
generate history charts of the selected finance metrics and desribe their trends.

Always be accurate. If you don't know the answer, say so.
"""

# Eearnings Call Agent
earnings_call_agent_system_message = """
You are a finance assistance that searches and summarizes the only last company earnings report and only from web search, 
for sales representative with focus on correlating facts with potential investments in AI/IT infrastructure returning the response 
in form of short statments/bullets. Ommit any side notes form search sources.

Be short and concise. If you cannot find a earnings call transcript
or report, say so.

Remove any links to the sources from the summary as this will be only text output in the textbox of the chatbot.

The example of the summary:
Company: Radiant Global Logistics Inc
* Very strong financial performance with 20% revenue growth year-over-year.
* Increased R&D expenses by 15%
* Increased CAPEX by 10%

Suggest to discuss where extra CAPEX and/or R&D investments can be made to improve the company's AI/IT infrastructure.

"""

earnings_call_agent_user_message = """
Summarize the most recent earnings call for {company_name} in year of {year}. 
"""


# Finance Metrics Agent
finance_metric_agent_system_message = """
You are a financial data assistant. Given a company name, a specific financial metric (e.g., revenue, net income, R&D, CAPEX), and a time window in years, extract a time series for that metric for the requested number of most recent years from up-to-date web sources.

Return the result as a JSON object with the following structure:
{
  "x": ["FY21Q4", "FY22Q1", ..., "FY25Q4"],  // x-axis labels, each as fiscal year and quarter
  "y": [123.4, 150.2, ..., 210.0],           // y-axis values, one per x label
  "x_label": "Fiscal Year & Quarter",
  "y_label": "Revenue (USD millions)"        // y_label should include the metric and its unit
}

Example for metric "revenue" for 5 years:
{
  "x": ["FY21Q4", "FY22Q1", "FY22Q2", "FY22Q3", "FY22Q4", "FY23Q1", "FY23Q2", "FY23Q3", "FY23Q4", "FY24Q1", "FY24Q2", "FY24Q3", "FY24Q4", "FY25Q1", "FY25Q2", "FY25Q3", "FY25Q4"],
  "y": [120.5, 130.2, 128.7, 135.0, 140.1, 145.3, 150.2, 155.0, 160.4, 165.0, 170.2, 175.1, 180.0, 185.5, 190.2, 200.0, 210.0],
  "x_label": "Fiscal Year & Quarter",
  "y_label": "Revenue (USD millions)"
}

If a value is missing, use null in the y array.
Return only valid JSON as described above.
"""
finance_metric_agent_user_message = """
Extract the time series for the company {company_name} for the metric {metric_name} for the last {years} years.
Return results  as a JSON object describe in system message. Avoid any extra text, explanations, or comments.
Return only valid JSON. Do not include any other text.
"""

# Tools

# Test variables
company_name = "Microsoft"
metric_name = "revenue"
years = 5  

In [13]:
class BaseLLMAgent(ABC):
    """
    Abstract base class for LLM-powered agents.
    """
    API_URL: str = SONAR_URL
    API_KEY: str = os.getenv("SONAR_API_KEY")
    HEADERS_TEMPLATE: Dict[str, str] = {
        "Authorization": "Bearer {api_key}",
        "Content-Type": "application/json",
    }
    SYSTEM_PROMPT: str = ""
    USER_PROMPT_TEMPLATE: str = ""
    MODEL: str = SONAR_MODEL

    def __init__(self, model_name: Optional[str] = None):
        self.model_name = model_name or self.MODEL

    @abstractmethod
    def build_user_prompt(self, *args, **kwargs) -> str:
        pass

    def _build_headers(self) -> Dict[str, str]:
        return {k: v.format(api_key=self.API_KEY) for k, v in self.HEADERS_TEMPLATE.items()}

    def _call_llm(self, user_prompt: str) -> Any:
        payload = {
            "model": self.model_name,
            "messages": [
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        }
        response = requests.post(self.API_URL, headers=self._build_headers(), json=payload)
        response.raise_for_status()
        return response.json()

class EarningsCallAgent(BaseLLMAgent):
    SYSTEM_PROMPT = earnings_call_agent_system_message
    USER_PROMPT_TEMPLATE = earnings_call_agent_user_message

    def build_user_prompt(self, company_name: str) -> str:
        return self.USER_PROMPT_TEMPLATE.format(
            company_name=company_name, year=datetime.datetime.now().year
        )

    def summarize_earnings_call(self, company_name: str) -> str:
        user_prompt = self.build_user_prompt(company_name)
        result = self._call_llm(user_prompt)
        return result.get("choices", [{}])[0].get("message", {}).get("content", "")

# # Summarize earnings call
# earnings_call_agent = EarningsCallAgent()
# summary = earnings_call_agent.summarize_earnings_call(account_name)
# print(summary)


In [16]:
def get_finance_metric(
    company_name: str,
    stock_symbol: str,
    frequency: str = "quarterly",
) -> str:
    return finnhub_client.financials_reported(
        symbol=stock_symbol,
        freq=frequency,
    )


fn_get_finance_metric = {
    "name": "get_finance_metric",
    "description": "Get reported financial metrics for a public company from finhub_client.financials_reported.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {
                "type": "string",
                "description": "The name of the company to get financial metrics for.",
            },
            "stock_symbol": {
                "type": "string",
                "description": "The stock symbol of the company, search on internet by company name if not provided.",
            },
            "frequency": {
                "type": "string",
                "enum": ["quarterly", "annual"],
                "default": "quarterly",
                "description": "Frequency of the financial report (quarterly or annual).",
            },
        },
        "required": ["company_name", "stock_symbol"],
        "additionalProperties": False,
    }
}

tools = [{ "type": "function", "function": fn_get_finance_metric}]

In [17]:
class ChatMessage(TypedDict):
    role: str
    content: str

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    company_name = arguments.get("company_name")
    stock_symbol = arguments.get("stock_symbol")
    frequency = arguments.get("frequency", "quarterly")
    financials = get_finance_metric(company_name, stock_symbol, frequency)
    response = {
        "role": "tool",
        "content": json.dumps({"financials": financials}),
        "tool_call_id": tool_call.id,
    }
    return response


def chat(history: List[ChatMessage]):
    messages = [{"role": "system", "content": chat_system_message}]
    messages.extend(history)
    response = openai.chat.completions.create(
        model=OPEN_AI_MODEL, messages=messages, tools=tools
    )
    # image = None

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        # image = artist(city)
        response = openai.chat.completions.create(
            model=OPEN_AI_MODEL, messages=messages
        )

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]
    
    # talker(reply)
    return history

user_message = f"Please provide me financial report for {company_name} for the last {years} years."
history = [{"role": "user", "content": user_message}]
history = chat(history)
print(history)

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_cjDsixK542IvNbTJMN2nWZks", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

In [ ]:
class ChatMessage(TypedDict):
    role: str
    content: str


def _build_assistant_tool_call_message(message):
    """
    Convert SDK message with tool_calls into a plain dict the API expects.
    {"role":"assistant","tool_calls":[{"id":...,"type":"function","function":{"name":...,"arguments":...}}]}
    """
    tc = message.tool_calls[0]
    return {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
        ],
    }


def _call_finance_metric_tool(message):
    """Execute the tool call and return a tool message dict."""
    tc = message.tool_calls[0]
    arguments = json.loads(tc.function.arguments)
    company_name = arguments.get("company_name")
    stock_symbol = arguments.get("stock_symbol")
    frequency = arguments.get("frequency", "quarterly")
    financials = get_finance_metric(company_name, stock_symbol, frequency)

    # Ensure content is a string (JSON). Some SDKs need default=str for dates/decimals
    return {
        "role": "tool",
        "content": json.dumps({"financials": financials}, default=str),
        "tool_call_id": tc.id,
    }


def chat(history: List[ChatMessage]):
    messages = [{"role": "system", "content": chat_system_message}]
    messages.extend(history)

    response = openai.chat.completions.create(
        model=OPEN_AI_MODEL,
        messages=messages,
        tools=tools,
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_tool_call_msg = _build_assistant_tool_call_message(
            response.choices[0].message
        )
        tool_result_msg = _call_finance_metric_tool(response.choices[0].message)

        # Append both the assistant tool_call and the tool result messages
        messages.append(assistant_tool_call_msg)
        messages.append(tool_result_msg)

        # Finalize with a follow-up completion
        response = openai.chat.completions.create(
            model=OPEN_AI_MODEL,
            messages=messages,
        )

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]
    return history


user_message = f"Please provide me financial report for {company_name} for the last {years} years."
history = [{"role": "user", "content": user_message}]
history = chat(history)
print(history)itities to json structure as described in system message from the user input: 
{user_prompt}.

The current date is: {current_date}.

Return only valid JSON. Do not include any other text.
"""

class FinanceMetricEntityRecognizer(BaseLLMAgent):
    SYSTEM_PROMPT = finance_metric_entity_recognizer_system_message
    USER_PROMPT_TEMPLATE = finance_metric_entity_recognizer_user_message

    def build_user_prompt(self, user_prompt: str) -> str:
        return self.USER_PROMPT_TEMPLATE.format(
            user_prompt=user_prompt, current_date=datetime.datetime.now().strftime("%Y-%m-%d")
        )

    def extract_entities(self, user_prompt: str) -> str:
        user_prompt = self.build_user_prompt(user_prompt)
        result = self._call_llm(user_prompt)
        return result.get("choices", [{}])[0].get("message", {}).get("content", "")
    
# Example usage of the FinanceMetricExtractor
finance_metric_entity_recognizer = FinanceMetricEntityRecognizer()
user_input = "Get capex for Apple Inc"
entities_json = finance_metric_entity_recognizer.extract_entities(user_input)
print(entities_json)

In [ ]:
import finnhub
import os

# Load API key from environment variable
api_key = os.getenv("FINNHUB_API_KEY")
finnhub_client = finnhub.Client(api_key=api_key)

# Get financials reported for a company (e.g., AAPL)
data = finnhub_client.financials_reported(symbol="AAPL", freq="quarterly")

# Extract revenue from each report
for report in data['data']:
    year = report.get('year')
    quarter = report.get('quarter')
    fiscal_period = f"FY{str(year)[-2:]}Q{quarter}"
    for metric in report['report']['ic']:
        if metric['concept'] == 'us-gaap_RevenueFromContractWithCustomerExcludingAssessedTax':
            print(f"Fiscal Period: {fiscal_period} → Revenue: ${metric['value']}")


In [ ]:
api_key = os.getenv("FINNHUB_API_KEY")
finnhub_client = finnhub.Client(api_key)
fr_data = finnhub_client.financials_reported(symbol="AAPL", freq="quarterly")
# f_data = finnhub_client.financials("AAPL", "bs", "quarterly")
fb_data = finnhub_client.company_basic_financials('AAPL', 'all')